In [13]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "David"
STUDENT_ID = "TUPM-26-1110"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: David TUPM-26-1110
Seed Digit: 0
ID Sum: 11
Random Seed: 261110
Verification Value: 55


In [14]:
# [CELL 2] LOAD SECOM AND IDENTIFY USABLE FEATURES

from pathlib import Path
import numpy as np

# LOCATE WORKSPACE AND DATA FOLDER
WORK_DIR = Path.cwd()

DATA_DIR = None

for folder in [WORK_DIR, *WORK_DIR.parents]:
    candidate = folder / "data"

    if (
        (candidate / "secom.data").is_file()
        and (candidate / "secom_labels.data").is_file()
    ):
        DATA_DIR = candidate
        break

assert DATA_DIR is not None, (
    "SECOM dataset not found. Make sure data/secom.data "
    "and data/secom_labels.data exist."
)

OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DEFINE DATASET PATHS
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

# LOAD DATA
raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

# VALIDATE DATA
assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

# IDENTIFY RECORDS AND FEATURES
n_records, n_features = raw_measurements.shape

feature_names = np.array([
    f"Feature_{i:03d}" for i in range(n_features)
])

outcomes = np.where(labels == -1, "Pass", "Fail")

# IDENTIFY MISSING VALUES
missing = np.isnan(raw_measurements)

observed = (~missing).sum(axis=0)

# IDENTIFY COMPLETELY MISSING FEATURES
all_missing = observed == 0

observed_min = np.where(
    missing,
    np.inf,
    raw_measurements
).min(axis=0)

observed_max = np.where(
    missing,
    -np.inf,
    raw_measurements
).max(axis=0)

# IDENTIFY CONSTANT FEATURES
constant = (
    (observed > 0) &
    (observed_min == observed_max)
)

# IDENTIFY USABLE FEATURES
usable_indices = np.flatnonzero(
    ~(all_missing | constant)
)

usable_features = feature_names[usable_indices]

# DISPLAY RESULTS
print("Data Folder:", DATA_DIR.resolve())
print("Measurement Matrix:", raw_measurements.shape)
print("Pass Records:", (labels == -1).sum(),
      "| Fail Records:", (labels == 1).sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

Data Folder: C:\Users\Rachel\OneDrive - Technological University of the Philippines\DAVID\Lab7_NumPy_Pandas_David\data
Measurement Matrix: (1567, 590)
Pass Records: 1463 | Fail Records: 104
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [15]:
# [CELL 3] FEATURE SELECTION AND NUMPY STATISTICS
n_select = 5 + (SEED_DIGIT % 3)

rng1 = np.random.default_rng(RANDOM_SEED)
selected_positions = rng1.choice(len(usable_indices), size=n_select, replace=False)
selected_indices = usable_indices[selected_positions]
selected_names = feature_names[selected_indices].tolist()

X = raw_measurements[:, selected_indices]

observed_counts = (~np.isnan(X)).sum(axis=0)
missing_counts = np.isnan(X).sum(axis=0)
missing_pct = 100 * missing_counts / X.shape[0]

means = np.nanmean(X, axis=0)
medians = np.nanmedian(X, axis=0)
sds = np.nanstd(X, axis=0, ddof=1)

print("Features To Select:", n_select)
print("Selected Features:", selected_names)
print("Means:", means)
print("Medians:", medians)
print("Sample SDs:", sds)
print("Missing Percentages:", missing_pct)

Features To Select: 5
Selected Features: ['Feature_040', 'Feature_435', 'Feature_586', 'Feature_572', 'Feature_147']
Means: [67.9049  9.3707  0.0215 28.4502  0.0174]
Medians: [78.29    4.5511  0.0205  8.65    0.0159]
Sample SDs: [24.0629 34.3698  0.0124 86.3047  0.0271]
Missing Percentages: [1.5316 0.1276 0.0638 0.     0.1276]


In [16]:
# [CELL 4] ROBUST STANDARDIZATION (MEDIAN AND MAD) AND FLAGS
mad = np.nanmedian(np.abs(X - medians), axis=0)
zero_mad = mad == 0
safe_mad = np.where(zero_mad, np.nan, mad)

with np.errstate(invalid="ignore", divide="ignore"):
    robust_z = 0.6745 * (X - medians) / safe_mad
    flags = np.abs(robust_z) > 3.5

flagged_counts = flags.sum(axis=0)
flagged_pct = 100 * flagged_counts / observed_counts
total_flagged = int(flags.sum())

print("MAD:", mad)
print("Zero-MAD Features:", int(zero_mad.sum()))
print("Flagged Counts:", flagged_counts)
print("Flagged Percentages:", flagged_pct)
print("Total Flagged Observations:", total_flagged)

MAD: [2.14   1.1519 0.0071 1.3    0.0056]
Zero-MAD Features: 0
Flagged Counts: [337  45  13  85  12]
Flagged Percentages: [21.8406  2.8754  0.8301  5.4244  0.7668]
Total Flagged Observations: 492


In [17]:
# [CELL 5] RANKING, CSV EXPORT AND VERIFICATION
measurement_quality = pd.DataFrame({
    "Feature": selected_names,
    "Observed": observed_counts,
    "Missing_Count": missing_counts,
    "Missing_Percent": missing_pct,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "MAD": mad,
    "Zero_MAD": zero_mad,
    "Flagged_Count": flagged_counts,
    "Flagged_Percent": flagged_pct,
})
measurement_quality = measurement_quality.sort_values(
    by=["Flagged_Percent", "Flagged_Count", "Feature"],
    ascending=[False, False, True]
).reset_index(drop=True)
measurement_quality.insert(0, "Rank", np.arange(1, len(measurement_quality) + 1))

quality_path = OUTPUT_DIR / "measurement_quality.csv"
measurement_quality.to_csv(quality_path, index=False)

check = pd.DataFrame(X, columns=selected_names)
stats_ok = (
    np.allclose(check.mean().to_numpy(), means)
    and np.allclose(check.median().to_numpy(), medians)
    and np.allclose(check.std(ddof=1).to_numpy(), sds)
    and np.allclose(check.isna().mean().to_numpy() * 100, missing_pct)
)

loop_flagged = []
for j in range(X.shape[1]):
    col = X[:, j]
    col = col[~np.isnan(col)]
    med = np.median(col)
    col_mad = np.median(np.abs(col - med))
    if col_mad == 0:
        loop_flagged.append(0)
    else:
        loop_flagged.append(int((np.abs(0.6745 * (col - med) / col_mad) > 3.5).sum()))
flags_ok = np.array_equal(loop_flagged, flagged_counts)
vectorized_verified = bool(stats_ok and flags_ok)

top = measurement_quality.iloc[0]
highest_missing = measurement_quality.loc[measurement_quality["Missing_Percent"].idxmax()]

print(measurement_quality.round(4))
print("--- Assessment Data: Exercise 1 ---")
print("Selected Features:", selected_names)
print("Highest Missing Percentage:", f"{highest_missing['Missing_Percent']:.2f}%", f"({highest_missing['Feature']})")
print("Feature with Highest Flagged Percentage:", top["Feature"], f"({top['Flagged_Percent']:.2f}%)")
print("Total Flagged Observations:", total_flagged)
print("Vectorized Results Verified:", vectorized_verified)
print(quality_path.name, "Created:", quality_path.is_file())

   Rank      Feature  Observed  Missing_Count  Missing_Percent     Mean  \
0     1  Feature_040      1543             24           1.5316  67.9049   
1     2  Feature_572      1567              0           0.0000  28.4502   
2     3  Feature_435      1565              2           0.1276   9.3707   
3     4  Feature_586      1566              1           0.0638   0.0215   
4     5  Feature_147      1565              2           0.1276   0.0174   

    Median  Sample_SD     MAD  Zero_MAD  Flagged_Count  Flagged_Percent  
0  78.2900    24.0629  2.1400     False            337          21.8406  
1   8.6500    86.3047  1.3000     False             85           5.4244  
2   4.5511    34.3698  1.1519     False             45           2.8754  
3   0.0205     0.0124  0.0071     False             13           0.8301  
4   0.0159     0.0271  0.0056     False             12           0.7668  
--- Assessment Data: Exercise 1 ---
Selected Features: ['Feature_040', 'Feature_435', 'Feature_586', 'Fea